Comparing reasearch.ipynb and modified research.ipnb then gpt provided this clean version of the code

##### well structured correct reasearch.ipynb notebook 

- Things to add
    - Visuals k means pca
    - data preprocessing steps
    - EDA
    - Outlier analysis

******************************************************************

************************************************************************


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Show plots inline
%matplotlib inline


In [4]:
# Adjust the path as needed
data = pd.read_csv("D:\\Projects_Final\\Diabetes_Prediction\\data\\diabetes.csv")
print("First 5 rows of data:")
print(data.head())

First 5 rows of data:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [5]:
cols_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_to_replace:
    median_val = data[col].median()
    data[col] = data[col].replace(0, median_val)

In [6]:
print("\nData description after zero replacement:")
print(data.describe())



Data description after zero replacement:
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  121.656250      72.386719      27.334635   94.652344   
std       3.369578   30.438286      12.096642       9.229014  105.547598   
min       0.000000   44.000000      24.000000       7.000000   14.000000   
25%       1.000000   99.750000      64.000000      23.000000   30.500000   
50%       3.000000  117.000000      72.000000      23.000000   31.250000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    32.450911                  0.471876   33.240885    0.348958  
std      6.875366                  0.331329   11.760232    0.47

In [7]:

X = data.drop('Outcome', axis=1)
y = data['Outcome']

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [9]:
# 6. PCA Transformations
# PCA - 3 components
pca_3 = PCA(n_components=3, random_state=42)
X3 = pca_3.fit_transform(X_scaled)

# PCA - 5 components
pca_5 = PCA(n_components=5, random_state=42)
X5 = pca_5.fit_transform(X_scaled)


In [10]:
print("\nExplained Variance by 3 Components:", pca_3.explained_variance_ratio_)
print("Explained Variance by 5 Components:", pca_5.explained_variance_ratio_)


Explained Variance by 3 Components: [0.27429654 0.20541242 0.13699051]
Explained Variance by 5 Components: [0.27429654 0.20541242 0.13699051 0.11091846 0.09803673]


In [11]:
# 7. KMeans Clustering
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
data['Cluster'] = clusters

print("\nCluster Distribution:")
print(data['Cluster'].value_counts())


Cluster Distribution:
Cluster
1    453
0    315
Name: count, dtype: int64


c:\Users\rakhi\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


In [12]:
# GLOBAL LIST TO COLLECT ALL METRICS
results_list = []

In [13]:
def compute_specificity(y_true, y_pred):
    """Compute specificity = TN / (TN + FP)."""
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2,2):
        TN, FP, FN, TP = cm.ravel()
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    else:
        specificity = 0
    return specificity

In [14]:
def model_pipeline(X_data, y_data, description):
    global results_list
    
    print(f"\n===============================")
    print(f"MODELING ON {description}")
    print(f"===============================")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_data,
        y_data,
        test_size=0.3,
        random_state=42,
        stratify=y_data
    )
    
    models = {
        "J48 Decision Tree": DecisionTreeClassifier(criterion='entropy', random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "Naïve Bayes": GaussianNB()
    }
    
    for model_name, model in models.items():
        # Fit model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Original prints
        print(f"\n--- {model_name} ---")
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))
        print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
        print("ROC AUC:", round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4))
        
        # Plot Decision Tree
        '''if model_name == "J48 Decision Tree":
            plt.figure(figsize=(12, 6))
            plot_tree(model, filled=True, feature_names=None)
            plt.title(f"Decision Tree - {description}")
            plt.show()'''
        
        # NEW METRICS FOR TABLE
        acc = accuracy_score(y_test, y_pred) * 100
        prec = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        specificity = compute_specificity(y_test, y_pred) * 100
        fscore = f1_score(y_test, y_pred, zero_division=0) * 100
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]) * 100
        
        # Save metrics to global list
        results_list.append({
            "Pipeline": description,
            "Model": model_name,
            "Accuracy (%)": round(acc, 2),
            "Precision (%)": round(prec, 2),
            "Sensitivity (%)": round(recall, 2),
            "Specificity (%)": round(specificity, 2),
            "F-score (%)": round(fscore, 2),
            "AUC (%)": round(auc, 2)
        })

In [15]:
results_list = []
model_pipeline(X_scaled, y, "Only Imputation (Original Features)")



MODELING ON Only Imputation (Original Features)

--- J48 Decision Tree ---
[[120  30]
 [ 31  50]]
              precision    recall  f1-score   support

           0       0.79      0.80      0.80       150
           1       0.62      0.62      0.62        81

    accuracy                           0.74       231
   macro avg       0.71      0.71      0.71       231
weighted avg       0.74      0.74      0.74       231

Accuracy: 0.7359
ROC AUC: 0.7086

--- Random Forest ---
[[130  20]
 [ 39  42]]
              precision    recall  f1-score   support

           0       0.77      0.87      0.82       150
           1       0.68      0.52      0.59        81

    accuracy                           0.74       231
   macro avg       0.72      0.69      0.70       231
weighted avg       0.74      0.74      0.74       231

Accuracy: 0.7446
ROC AUC: 0.8204

--- Naïve Bayes ---
[[121  29]
 [ 32  49]]
              precision    recall  f1-score   support

           0       0.79      0.81   

In [16]:
model_pipeline(X3, y, "Feature Selection (3-Factor)")


MODELING ON Feature Selection (3-Factor)

--- J48 Decision Tree ---
[[117  33]
 [ 33  48]]
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       150
           1       0.59      0.59      0.59        81

    accuracy                           0.71       231
   macro avg       0.69      0.69      0.69       231
weighted avg       0.71      0.71      0.71       231

Accuracy: 0.7143
ROC AUC: 0.6863

--- Random Forest ---
[[122  28]
 [ 37  44]]
              precision    recall  f1-score   support

           0       0.77      0.81      0.79       150
           1       0.61      0.54      0.58        81

    accuracy                           0.72       231
   macro avg       0.69      0.68      0.68       231
weighted avg       0.71      0.72      0.71       231

Accuracy: 0.7186
ROC AUC: 0.7666

--- Naïve Bayes ---
[[125  25]
 [ 43  38]]
              precision    recall  f1-score   support

           0       0.74      0.83      0.79

In [17]:
model_pipeline(X5, y, "Feature Selection (5-Factor)")


MODELING ON Feature Selection (5-Factor)

--- J48 Decision Tree ---
[[118  32]
 [ 37  44]]
              precision    recall  f1-score   support

           0       0.76      0.79      0.77       150
           1       0.58      0.54      0.56        81

    accuracy                           0.70       231
   macro avg       0.67      0.66      0.67       231
weighted avg       0.70      0.70      0.70       231

Accuracy: 0.7013
ROC AUC: 0.6649

--- Random Forest ---
[[124  26]
 [ 34  47]]
              precision    recall  f1-score   support

           0       0.78      0.83      0.81       150
           1       0.64      0.58      0.61        81

    accuracy                           0.74       231
   macro avg       0.71      0.70      0.71       231
weighted avg       0.74      0.74      0.74       231

Accuracy: 0.7403
ROC AUC: 0.7815

--- Naïve Bayes ---
[[120  30]
 [ 41  40]]
              precision    recall  f1-score   support

           0       0.75      0.80      0.77

In [18]:
# ==========================================
# Print Tables Like Tables 13–15
# ==========================================

# Convert list to DataFrame
df_all_results = pd.DataFrame(results_list)

# Create separate tables
for pipeline_name in df_all_results["Pipeline"].unique():
    df_subset = df_all_results[df_all_results["Pipeline"] == pipeline_name]
    print(f"\n===== RESULTS TABLE: {pipeline_name} =====\n")
    print(df_subset.drop(columns=["Pipeline"]).to_string(index=False))



===== RESULTS TABLE: Only Imputation (Original Features) =====

            Model  Accuracy (%)  Precision (%)  Sensitivity (%)  Specificity (%)  F-score (%)  AUC (%)
J48 Decision Tree         73.59          62.50            61.73            80.00        62.11    70.86
    Random Forest         74.46          67.74            51.85            86.67        58.74    82.04
      Naïve Bayes         73.59          62.82            60.49            80.67        61.64    79.63

===== RESULTS TABLE: Feature Selection (3-Factor) =====

            Model  Accuracy (%)  Precision (%)  Sensitivity (%)  Specificity (%)  F-score (%)  AUC (%)
J48 Decision Tree         71.43          59.26            59.26            78.00        59.26    68.63
    Random Forest         71.86          61.11            54.32            81.33        57.52    76.66
      Naïve Bayes         70.56          60.32            46.91            83.33        52.78    76.98

===== RESULTS TABLE: Feature Selection (5-Factor) ==

- ## User input and prediction

In [ ]:
feature_ranges = {
    "Pregnancies": (0, 20),            
    "Glucose": (0, 199),
    "BloodPressure": (0, 122),
    "SkinThickness": (0, 99),
    "Insulin": (0, 846),
    "BMI": (0, 67.1),
    "DiabetesPedigreeFunction": (0.0, 2.5),  
    "Age": (1, 120)                   
}

# Function to get user input safely
def get_user_input():
    user_data = {}
    print("\nEnter patient details below:")

    for feature, (min_val, max_val) in feature_ranges.items():
        while True:
            try:
                val = float(input(f"{feature} (Allowed Range: {min_val} to {max_val}): "))
                if min_val <= val <= max_val:
                    user_data[feature] = val
                    break
                else:
                    print(f" Value out of range! Must be between {min_val} and {max_val}. Please try again.")
            except ValueError:
                print("Invalid input. Please enter a number.")
    
    return pd.DataFrame([user_data])

# Take user input
user_df = get_user_input()

print("\n User Input Received:")
print(user_df)

# Replace zeros with median (for relevant columns)
for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
    median_val = data[col].median()
    user_df[col] = user_df[col].replace(0, median_val)

# Scale the input
user_scaled = scaler.transform(user_df)

# Predict using Random Forest
best_model = RandomForestClassifier(random_state=42)
best_model.fit(X_scaled, y)
user_pred = best_model.predict(user_scaled)
user_prob = best_model.predict_proba(user_scaled)[0][1]

# Output the prediction
print("\n===============================")
if user_pred[0] == 1:
    print(f" **Prediction:** The patient is likely to have diabetes.")
else:
    print(f" **Prediction:** The patient is NOT likely to have diabetes.")
print(f"Predicted Probability of Diabetes: {round(user_prob*100, 2)}%")
print("===============================")



Enter patient details below:
 Value out of range! Must be between 0 and 846. Please try again.

 User Input Received:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0          2.0    150.0          120.0           56.0    845.0  56.0   

   DiabetesPedigreeFunction    Age  
0                       1.0  115.0  

 **Prediction:** The patient is likely to have diabetes.
Predicted Probability of Diabetes: 60.0%
